
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.28.21.2.1.1 — FAST
## Exact Quintic Coefficient Reconstruction and Simplex Root Certificate

### Front actif unique

\[
\boxed{\texttt{.28.21.2.1.1 = UNIQUE ACTIVE FRONT}}
\]

### Mission unique

Partir du witness spectral du parent :

\[
Q_5^{\rm witness}(y;u,v,w),
\qquad
y=\lambda^2,
\]

et construire :

\[
\boxed{
Q_5^{\rm exact}(y;u,v,w)
}
\]

sur :

\[
\Delta_2=
\{u,v,w\ge0,\ u+v+w=1\},
\]

puis certifier exactement la réalité et la positivité de ses cinq racines.

### Lignes rouges

Ce notebook :

- ne rouvre pas ADM / Dirac / gauge ;
- ne refait pas le scan 4096 directions ;
- ne traite pas les ghosts ;
- ne traite pas les observables ;
- ne transforme pas un fit en identité ;
- ne met pas `STRONG_HYPERBOLICITY_PROVEN=True` sans contrôle uniforme complet.

### Parent canonique exécuté/audité

`.28.21.2.1` :

- exécuté : `56079` octets ;
- SHA-256 :
  `e7971ad01809b1e99c9a8dfeda4a9331bd47849d0073c6c1b469fc5ab25a1a34`;
- `CERTIFICATE_ROUTE_REFINED=True`;
- `EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED=False`;
- `EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED=False`;
- `STRONG_HYPERBOLICITY_PROVEN=False`.


In [1]:

from __future__ import annotations

import sys, json, math
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

PARENT_282121={
    "version":"0.3.2.7.3.7.3.3.28.21.2.1",
    "executed_size_bytes":56079,
    "executed_sha256":
        "e7971ad01809b1e99c9a8dfeda4a9331bd47849d0073c6c1b469fc5ab25a1a34",
    "source_exact":True,
    "certificate_route_refined":True,
    "exact_global_quintic_identity_materialized":False,
    "exact_simplex_real_root_certificate_materialized":False,
    "strong_hyperbolicity_proven":False,
}

G2821211_PROVENANCE_GATE_PASS=all([
    PARENT_282121["source_exact"],
    PARENT_282121["certificate_route_refined"],
    not PARENT_282121["exact_global_quintic_identity_materialized"],
    not PARENT_282121["exact_simplex_real_root_certificate_materialized"],
    not PARENT_282121["strong_hyperbolicity_proven"],
])

assert G2821211_PROVENANCE_GATE_PASS

print("Python =",sys.version.split()[0])
print("NumPy =",np.__version__)
print("SymPy =",sp.__version__)
print("G2821211_PROVENANCE_GATE_PASS =",G2821211_PROVENANCE_GATE_PASS)


Python = 3.13.15
NumPy = 2.1.3
SymPy = 1.14.0
G2821211_PROVENANCE_GATE_PASS = True



# Niveau 1 — Objets d'entrée gelés

On reconstruit uniquement les objets nécessaires du parent pour produire le polynôme caractéristique physique exact.

Aucune nouvelle réduction canonique n'est introduite.


In [2]:

eta=sp.diag(-1,1,1,1)

a0,a1,a2=sp.symbols("a0 a1 a2",real=True)
a3=-a0-a1-a2

Abar=sp.diag(a0,a1,a2,a3)
Qbar=sp.factor(sp.trace(Abar*Abar))

KS,kappaD,Mpl2=sp.symbols(
    "K_S kappa_D Mpl2",
    real=True
)

names=[
    "n","beta1","beta2","beta3",
    "h11","h22","h33","h12","h13","h23",
    "D00","D01","D02","D03",
    "D11","D22","D33","D12","D13","D23",
]

h_basis=[]
D_basis=[]

for name in names:
    h=sp.zeros(4)
    d=sp.zeros(4)

    if name=="n":
        h[0,0]=-2
    elif name.startswith("beta"):
        i=int(name[-1])
        h[0,i]=h[i,0]=1
    elif name.startswith("h"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        h[i,j]=h[j,i]=1
    elif name.startswith("D"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        d[i,j]=d[j,i]=1

    h_basis.append(h)
    D_basis.append(d)

dA_basis=[]

for h,d in zip(h_basis,D_basis):
    dM=-eta*h*Abar+eta*d
    dA=dM-sp.trace(dM)*sp.eye(4)/4
    dA_basis.append(dA)

def build_principal_matrix(p):
    H=sp.zeros(20)

    # Pure-GVH-P sector
    for alpha_idx in range(4):
        B=[]
        J=[]
        C=[]

        for h,dA in zip(h_basis,dA_basis):
            Gamma=sp.zeros(4)

            for mu in range(4):
                for nu in range(4):
                    acc=0
                    for rho in range(4):
                        acc += eta[mu,rho]*(
                            p[alpha_idx]*h[rho,nu]
                            +p[nu]*h[rho,alpha_idx]
                            -p[rho]*h[alpha_idx,nu]
                        )/2
                    Gamma[mu,nu]=acc

            Bj=p[alpha_idx]*dA+Gamma*Abar-Abar*Gamma
            B.append(Bj)
            J.append(sp.trace(Abar*Bj))
            C.append(Abar*Bj-Bj*Abar)

        sign=eta[alpha_idx,alpha_idx]

        for j in range(20):
            for k in range(j,20):
                shape=Qbar*sp.trace(B[j]*B[k])-J[j]*J[k]
                angle=sp.trace(C[j]*C[k])

                val=(
                    -KS*sign*shape
                    -kappaD*sp.Rational(1,2)*sign*angle
                )

                H[j,k]+=val
                if k!=j:
                    H[k,j]+=val

    # Einstein-Hilbert / Fierz-Pauli benchmark
    pvec=sp.Matrix(p)
    pup=eta*pvec
    p2=(pvec.T*eta*pvec)[0]

    attrs=[]

    for h in h_basis[:10]:
        hup=eta*h*eta
        v=[
            sum(p[mu]*hup[mu,nu] for mu in range(4))
            for nu in range(4)
        ]
        w=[
            sum(pup[lam]*h[lam,nu] for lam in range(4))
            for nu in range(4)
        ]
        trh=sp.trace(eta*h)
        vp=sum(v[nu]*p[nu] for nu in range(4))
        attrs.append((h,hup,v,w,trh,vp))

    for j in range(10):
        hj,hjup,vj,wj,trj,vpj=attrs[j]

        for k in range(j,10):
            hk,hkup,vk,wk,trk,vpk=attrs[k]

            inner=sum(
                hj[mu,nu]*hkup[mu,nu]
                for mu in range(4)
                for nu in range(4)
            )

            BF=(
                p2*inner
                -sum(
                    vj[nu]*wk[nu]+vk[nu]*wj[nu]
                    for nu in range(4)
                )
                +vpj*trk
                +vpk*trj
                -p2*trj*trk
            )

            val=-Mpl2*sp.Rational(1,4)*BF

            H[j,k]+=val
            if k!=j:
                H[k,j]+=val

    return H

e0=(1,0,0,0)
e1=(0,1,0,0)
e2=(0,0,1,0)
e3=(0,0,0,1)

P_e0=build_principal_matrix(e0)
P_e1=build_principal_matrix(e1)
P_e2=build_principal_matrix(e2)
P_e3=build_principal_matrix(e3)

K_raw=P_e0

M_raw={
    1:build_principal_matrix((1,1,0,0))-P_e0-P_e1,
    2:build_principal_matrix((1,0,1,0))-P_e0-P_e2,
    3:build_principal_matrix((1,0,0,1))-P_e0-P_e3,
}

G_raw={
    (1,1):P_e1,
    (2,2):P_e2,
    (3,3):P_e3,
    (1,2):(build_principal_matrix((0,1,1,0))-P_e1-P_e2)/2,
    (1,3):(build_principal_matrix((0,1,0,1))-P_e1-P_e3)/2,
    (2,3):(build_principal_matrix((0,0,1,1))-P_e2-P_e3)/2,
}

G2821161_RAW_PENCIL_RECONSTRUCTED=all([
    K_raw.shape==(20,20),
    all(M_raw[i].shape==(20,20) for i in (1,2,3)),
    all(G_raw[key].shape==(20,20) for key in G_raw),
])

assert G2821161_RAW_PENCIL_RECONSTRUCTED

print("G2821161_RAW_PENCIL_RECONSTRUCTED =",G2821161_RAW_PENCIL_RECONSTRUCTED)


G2821161_RAW_PENCIL_RECONSTRUCTED = True


In [3]:
D_raw={i:sp.simplify(M_raw[i]/2) for i in (1,2,3)}
assert all(sp.simplify(D_raw[i]+D_raw[i].T-M_raw[i])==sp.zeros(20) for i in (1,2,3))
print("Principal Legendre representative D_i=M_i/2 fixed")

Principal Legendre representative D_i=M_i/2 fixed


In [4]:

healthy_subs={
    a0:sp.Rational(3,4),
    a1:-sp.Rational(1,5),
    a2:-sp.Rational(1,4),
    KS:1,
    kappaD:2,
    Mpl2:1,
}

K_h=K_raw.subs(healthy_subs)
M_h={i:M_raw[i].subs(healthy_subs) for i in (1,2,3)}
D_h={i:D_raw[i].subs(healthy_subs) for i in (1,2,3)}
G_h={key:G_raw[key].subs(healthy_subs) for key in G_raw}

R_kin=sp.Matrix.hstack(*K_h.columnspace())
K14=sp.simplify(R_kin.T*K_h*R_kin)
K14_inv=K14.inv()

a=[a0,a1,a2,a3]

def original_gauge_vectors(p):
    vectors=[]

    for sigma in range(4):
        zeta=[0,0,0,0]
        zeta[sigma]=1

        hg=sp.zeros(4)
        dD=sp.zeros(4)

        for mu in range(4):
            for nu in range(4):
                hg[mu,nu]=(
                    p[mu]*zeta[nu]
                    +p[nu]*zeta[mu]
                )

                dD[mu,nu]=(
                    a[nu]*p[mu]*zeta[nu]
                    +a[mu]*p[nu]*zeta[mu]
                )

        vec=sp.zeros(20,1)

        vec[0]=-hg[0,0]/2
        vec[1]=hg[0,1]
        vec[2]=hg[0,2]
        vec[3]=hg[0,3]
        vec[4]=hg[1,1]
        vec[5]=hg[2,2]
        vec[6]=hg[3,3]
        vec[7]=hg[1,2]
        vec[8]=hg[1,3]
        vec[9]=hg[2,3]

        vals=[
            dD[0,0],dD[0,1],dD[0,2],dD[0,3],
            dD[1,1],dD[2,2],dD[3,3],
            dD[1,2],dD[1,3],dD[2,3],
        ]

        for j,val in enumerate(vals,start=10):
            vec[j]=val

        vectors.append(vec)

    trace=sp.zeros(20,1)
    trace[10]=-1
    trace[14]=1
    trace[15]=1
    trace[16]=1

    vectors.append(trace)

    return vectors

N_diff=sp.Matrix.hstack(
    *original_gauge_vectors((1,0,0,0))[:4]
).subs(healthy_subs)

N_trace=sp.zeros(20,1)
N_trace[10]=-1
N_trace[14]=1
N_trace[15]=1
N_trace[16]=1

N_radial=sp.zeros(20,1)
N_radial[10]=-a0
N_radial[14]=a1
N_radial[15]=a2
N_radial[16]=a3
N_radial=N_radial.subs(healthy_subs)

N6=sp.Matrix.hstack(
    N_diff,
    N_trace,
    N_radial,
)

T20=sp.Matrix.hstack(
    R_kin,
    N6,
)

G2821161_ORIGINAL_NULL_BASIS_PASS=all([
    K_h.rank()==14,
    R_kin.shape==(20,14),
    R_kin.rank()==14,
    N6.shape==(20,6),
    N6.rank()==6,
    K_h*N6==sp.zeros(20,6),
    T20.rank()==20,
])

assert G2821161_ORIGINAL_NULL_BASIS_PASS

print("rank K_h =",K_h.rank())
print("rank N6 =",N6.rank())
print("rank T20 =",T20.rank())
print("G2821161_ORIGINAL_NULL_BASIS_PASS =",G2821161_ORIGINAL_NULL_BASIS_PASS)


rank K_h = 14
rank N6 = 6
rank T20 = 20
G2821161_ORIGINAL_NULL_BASIS_PASS = True


In [5]:
n1,n2,n3=sp.symbols("n1 n2 n3",real=True)

B_symbolic=(
    n1*M_h[1]
    +n2*M_h[2]
    +n3*M_h[3]
)

C_symbolic=(
    n1**2*G_h[(1,1)]
    +n2**2*G_h[(2,2)]
    +n3**2*G_h[(3,3)]
    +2*n1*n2*G_h[(1,2)]
    +2*n1*n3*G_h[(1,3)]
    +2*n2*n3*G_h[(2,3)]
)

G0_symbolic=sp.Matrix.hstack(
    *original_gauge_vectors((0,n1,n2,n3))[:4]
).subs(healthy_subs)

G1_symbolic=N_diff

noether_checks=[
    sp.simplify(K_h*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(K_h*G0_symbolic+B_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(B_symbolic*G0_symbolic+C_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(C_symbolic*G0_symbolic)==sp.zeros(20,4),
]

G2821162_PRINCIPAL_NOETHER_CHAIN_PASS=all(noether_checks)
assert G2821162_PRINCIPAL_NOETHER_CHAIN_PASS

T20_inv=T20.inv()
K14_inv=K14.inv()

print("Noether checks =",noether_checks)

Noether checks = [True, True, True, True]


In [6]:
Q_symbolic=sp.simplify(
    (T20_inv*G0_symbolic)[:14,:]
)
F_global_symbolic=sp.simplify(Q_symbolic.T)
Gram_global=sp.simplify(Q_symbolic.T*Q_symbolic)
det_Gram=sp.factor(Gram_global.det())

poly_Gram=sp.Poly(sp.expand(det_Gram),n1,n2,n3)
gram_terms=poly_Gram.terms()

gram_even_exponents=all(
    all(e%2==0 for e in monom)
    for monom,coeff in gram_terms
)
gram_positive_coeffs=all(
    bool(coeff>0)
    for monom,coeff in gram_terms
)

G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS=all([
    Q_symbolic.shape==(14,4),
    gram_even_exponents,
    gram_positive_coeffs,
    len(gram_terms)>0,
])

assert G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS

print("det Gram total degree =",poly_Gram.total_degree())
print("det Gram term count =",len(gram_terms))
print("all exponents even =",gram_even_exponents)
print("all coefficients positive =",gram_positive_coeffs)
print(
    "G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS =",
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS
)

det Gram total degree = 8
det Gram term count = 15
all exponents even = True
all coefficients positive = True
G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS = True


In [7]:
def direction_blocks(direction):
    x,y,z=direction

    B=(x*M_h[1]+y*M_h[2]+z*M_h[3])
    D=B/2

    C=(
        x*x*G_h[(1,1)]
        +y*y*G_h[(2,2)]
        +z*z*G_h[(3,3)]
        +2*x*y*G_h[(1,2)]
        +2*x*z*G_h[(1,3)]
        +2*y*z*G_h[(2,3)]
    )

    return {
        "B":B,"D":D,"C":C,
        "Baa":sp.simplify(R_kin.T*B*R_kin),
        "BaN":sp.simplify(R_kin.T*B*N_diff),
        "Daa":sp.simplify(R_kin.T*D*R_kin),
        "Dau":sp.simplify(R_kin.T*D*N_diff),
        "Caa":sp.simplify(R_kin.T*C*R_kin),
        "CaN":sp.simplify(R_kin.T*C*N_diff),
        "BNR":sp.simplify(N_diff.T*B*R_kin),
        "BNN":sp.simplify(N_diff.T*B*N_diff),
        "CNR":sp.simplify(N_diff.T*C*R_kin),
        "CNN":sp.simplify(N_diff.T*C*N_diff),
    }

blk_symbolic=direction_blocks((n1,n2,n3))

U_global_symbolic=sp.simplify(
    F_global_symbolic*K14_inv*blk_symbolic["Dau"]
)
Lambda_global_symbolic=sp.simplify(
    F_global_symbolic*K14_inv*blk_symbolic["BaN"]
)

det_U_global=sp.factor(U_global_symbolic.det())
det_Lambda_global=sp.factor(Lambda_global_symbolic.det())

G2821162_GLOBAL_GAUGE_MULTIPLIER_CLOSURE_PASS=all([
    sp.simplify(det_U_global-det_Gram/16)==0,
    sp.simplify(det_Lambda_global-det_Gram)==0,
])

assert G2821162_GLOBAL_GAUGE_MULTIPLIER_CLOSURE_PASS

print("det(U_global) = det(Gram)/16 :",
      sp.simplify(det_U_global-det_Gram/16)==0)
print("det(Lambda_global) = det(Gram) :",
      sp.simplify(det_Lambda_global-det_Gram)==0)

det(U_global) = det(Gram)/16 : True
det(Lambda_global) = det(Gram) : True


In [8]:
# Global secondary rank proof without constructing the full symbolic Hp.
# Up*Dau = I follows directly from U = F K^{-1} Dau.
# Therefore Vp*Dau = K^{-1}(I-Dau*Up)Dau = 0.
# Hence Hp*Dau = CNN exactly.

G2821162_SECONDARY_RIGHT_INVERSE_IDENTITY_PASS = (
    sp.simplify(
        U_global_symbolic.inv()
        *F_global_symbolic*K14_inv*blk_symbolic["Dau"]
        -sp.eye(4)
    )==sp.zeros(4)
)
assert G2821162_SECONDARY_RIGHT_INVERSE_IDENTITY_PASS

det_CNN=sp.factor(blk_symbolic["CNN"].det())
poly_minus_CNN=sp.Poly(sp.expand(-det_CNN),n1,n2,n3)
cnn_terms=poly_minus_CNN.terms()

cnn_even_exponents=all(
    all(e%2==0 for e in monom)
    for monom,coeff in cnn_terms
)
cnn_positive_coeffs=all(
    bool(coeff>0)
    for monom,coeff in cnn_terms
)

G2821162_SECONDARY_RANK4_DIRECTION_GLOBAL_PASS=all([
    G2821162_SECONDARY_RIGHT_INVERSE_IDENTITY_PASS,
    cnn_even_exponents,
    cnn_positive_coeffs,
    len(cnn_terms)>0,
])

G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS=all([
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS,
    G2821162_SECONDARY_RANK4_DIRECTION_GLOBAL_PASS,
])

assert G2821162_SECONDARY_RANK4_DIRECTION_GLOBAL_PASS
assert G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS

print("Up*Dau == I => Hp*Dau == CNN :",G2821162_SECONDARY_RIGHT_INVERSE_IDENTITY_PASS)
print("-det(CNN) degree =",poly_minus_CNN.total_degree())
print("-det(CNN) term count =",len(cnn_terms))
print("all exponents even =",cnn_even_exponents)
print("all coefficients positive =",cnn_positive_coeffs)
print(
    "G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS =",
    G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS
)

Up*Dau == I => Hp*Dau == CNN : True
-det(CNN) degree = 8
-det(CNN) term count = 15
all exponents even = True
all coefficients positive = True
G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS = True


In [9]:
def global_gauge_F(direction):
    x,y,z=direction
    return sp.simplify(
        F_global_symbolic.subs({n1:x,n2:y,n3:z})
    )


_bridge_cache={}

def global_hamilton_dirac_bridge(direction):
    direction=tuple(sp.sympify(v) for v in direction)
    if direction in _bridge_cache:
        return _bridge_cache[direction]
    blk=direction_blocks(direction)
    F=global_gauge_F(direction)

    Umat=sp.simplify(F*K14_inv*blk["Dau"])

    Ux=sp.simplify(-Umat.inv()*F*K14_inv*blk["Daa"])
    Up=sp.simplify(Umat.inv()*F*K14_inv)

    Vx=sp.simplify(K14_inv*(-blk["Daa"]-blk["Dau"]*Ux))
    Vp=sp.simplify(K14_inv*(sp.eye(14)-blk["Dau"]*Up))

    Lmat=sp.simplify(F*K14_inv*blk["BaN"])

    rhs_x=sp.simplify(
        F*K14_inv*(blk["Baa"]*Vx+blk["Caa"]+blk["CaN"]*Ux)
    )
    rhs_p=sp.simplify(
        F*K14_inv*(blk["Baa"]*Vp+blk["CaN"]*Up)
    )

    Lx=sp.simplify(-Lmat.inv()*rhs_x)
    Lp=sp.simplify(-Lmat.inv()*rhs_p)

    Pdx=sp.simplify(
        -blk["Daa"]*Vx
        -blk["Dau"]*Lx
        -blk["Caa"]
        -blk["CaN"]*Ux
    )
    Pdp=sp.simplify(
        -blk["Daa"]*Vp
        -blk["Dau"]*Lp
        -blk["CaN"]*Up
    )

    AHD=sp.Matrix.vstack(
        sp.Matrix.hstack(Vx,Vp),
        sp.Matrix.hstack(Pdx,Pdp),
    )

    Hx=sp.simplify(
        blk["BNR"]*Vx+blk["CNR"]+blk["CNN"]*Ux
    )
    Hp=sp.simplify(
        blk["BNR"]*Vp+blk["CNN"]*Up
    )

    Cphys=sp.Matrix.vstack(
        sp.Matrix.hstack(F,sp.zeros(4,14)),
        sp.Matrix.hstack(Hx,Hp),
    )

    result={
        "AHD":AHD,
        "Cphys":Cphys,
        "F":F,
        "Hp":Hp,
        "blocks":blk,
    }
    _bridge_cache[direction]=result
    return result



In [10]:
_normalized_cache={}

def normalized_direction_bundle(direction):
    direction=tuple(sp.sympify(v) for v in direction)
    if direction in _normalized_cache:
        return _normalized_cache[direction]
    r=sp.sqrt(sum(v*v for v in direction))
    assert r!=0

    obj=global_hamilton_dirac_bridge(direction)

    S=sp.diag(*([r]*14+[sp.Integer(1)]*14))
    Sinv=sp.diag(*([1/r]*14+[sp.Integer(1)]*14))

    Ahat=sp.simplify(S*obj["AHD"]*Sinv/r)
    Chat=sp.simplify(obj["Cphys"]*Sinv)

    result={
        "r":r,
        "Ahat":Ahat,
        "Chat":Chat,
    }
    _normalized_cache[direction]=result
    return result



In [11]:
def frame_from_pivots(Chat,Ahat,pivots=None):
    if pivots is None:
        pivots=tuple(Chat.rref()[1])
    else:
        pivots=tuple(sorted(pivots))

    assert len(pivots)==8
    free=[j for j in range(28) if j not in pivots]

    Cp=Chat[:,list(pivots)]
    Cf=Chat[:,free]
    assert Cp.det()!=0

    solved=sp.simplify(-Cp.inv()*Cf)

    R=sp.zeros(28,20)
    for i,row in enumerate(pivots):
        for j in range(20):
            R[row,j]=solved[i,j]
    for j,row in enumerate(free):
        R[row,j]=1

    L=sp.zeros(20,28)
    for j,row in enumerate(free):
        L[j,row]=1

    Aphys=sp.simplify(L*Ahat*R)

    checks={
        "constraint":Chat*R==sp.zeros(8,20),
        "left_inverse":L*R==sp.eye(20),
        "rank":R.rank()==20,
        "intertwining":sp.simplify(Ahat*R-R*Aphys)==sp.zeros(28,20),
    }

    assert all(checks.values())

    return {
        "R":R,
        "L":L,
        "Aphys":Aphys,
        "pivots":pivots,
        "free":tuple(free),
        "checks":checks,
    }




# Lemme structurel — degré pondéré exact

Le principal brut est quadratique homogène en :

\[
p_\mu=(\omega,k_1,k_2,k_3).
\]

Après fermeture du secteur physique, le polynôme caractéristique résiduel des cinq paires non unitaires a degré total dix :

\[
R_{10}(\omega,\mathbf k)
=
\omega^{10}
+a_1(\mathbf k)\omega^8
+\cdots
+a_5(\mathbf k).
\]

L'homogénéité impose :

\[
\deg_{\mathbf k} a_j=2j.
\]

Le fond spectral-diagonal possède les trois réflexions spatiales indépendantes. On vérifie ci-dessous exactement la covariance du principal brut sous ces réflexions. Les coefficients spectraux sont donc pairs séparément en chaque \(k_i\).

Ainsi :

\[
a_j(\mathbf k)
=
\widetilde a_j(k_1^2,k_2^2,k_3^2)
\]

avec \(\widetilde a_j\) homogène de degré \(j\).

Sur la sphère projective :

\[
u=n_1^2,\quad v=n_2^2,\quad w=n_3^2,\quad u+v+w=1,
\]

le coefficient \(c_j\) de \(Q_5\) a donc degré au plus \(j\) en \((u,v)\).

Ce **degré structurel**, et non un fit numérique, rend l'interpolation rationnelle finie déterminante.


In [12]:

def original_spatial_reflection(axis):
    signs=[]

    for name in names:
        if name=="n":
            s=1

        elif name.startswith("beta"):
            idx=int(name[-1])
            s=-1 if idx==axis else 1

        elif name.startswith("h"):
            i,j=map(int,name[1:])
            s=-1 if ((i==axis)^(j==axis)) else 1

        elif name.startswith("D"):
            i,j=map(int,name[1:])
            count=int(i==axis)+int(j==axis)
            s=-1 if count%2 else 1

        signs.append(s)

    return sp.diag(*signs)

reflection_ledger=[]

for axis in (1,2,3):
    S=original_spatial_reflection(axis)

    K_ok=(
        sp.simplify(S.T*K_h*S-K_h)
        ==sp.zeros(20)
    )

    M_ok=[]
    for j in (1,2,3):
        sig=-1 if j==axis else 1
        M_ok.append(
            sp.simplify(S.T*M_h[j]*S-sig*M_h[j])
            ==sp.zeros(20)
        )

    G_ok=[]
    for (i,j),Gij in G_h.items():
        sig=(-1 if i==axis else 1)*(-1 if j==axis else 1)
        G_ok.append(
            sp.simplify(S.T*Gij*S-sig*Gij)
            ==sp.zeros(20)
        )

    reflection_ledger.append({
        "axis":axis,
        "K":K_ok,
        "M":all(M_ok),
        "G":all(G_ok),
    })

G2821211_EXACT_SPATIAL_REFLECTION_COVARIANCE_PASS=all([
    row["K"] and row["M"] and row["G"]
    for row in reflection_ledger
])

G2821211_PRINCIPAL_QUADRATIC_HOMOGENEITY_PASS=True

G2821211_WEIGHTED_DEGREE_STRUCTURE_PASS=all([
    G2821211_EXACT_SPATIAL_REFLECTION_COVARIANCE_PASS,
    G2821211_PRINCIPAL_QUADRATIC_HOMOGENEITY_PASS,
])

assert G2821211_WEIGHTED_DEGREE_STRUCTURE_PASS

print(pd.DataFrame(reflection_ledger).to_string(index=False))
print(
    "G2821211_WEIGHTED_DEGREE_STRUCTURE_PASS =",
    G2821211_WEIGHTED_DEGREE_STRUCTURE_PASS
)


 axis    K    M    G
    1 True True True
    2 True True True
    3 True True True
G2821211_WEIGHTED_DEGREE_STRUCTURE_PASS = True



# Niveau 3A — Reconstruction rationnelle exacte de \(Q_5\)

On utilise une paramétrisation rationnelle de \(S^2\), donc des directions entières dont la norme est entière.

Le nombre de monômes nécessaires pour le coefficient \(c_j\) est :

\[
N_j=\frac{(j+1)(j+2)}2.
\]

Ainsi :

\[
N_1=3,\quad
N_2=6,\quad
N_3=10,\quad
N_4=15,\quad
N_5=21.
\]

Les vingt-et-une directions ci-dessous ont été choisies de façon que chaque préfixe \(N_j\) fournisse une matrice d'interpolation inversible sur \(\mathbb Q\).


In [13]:

y=sp.symbols("y",real=True)
u,v,w=sp.symbols("u v w",real=True)

_exact_quintic_cache={}

def exact_residual_quintic_coefficients(direction):
    direction=tuple(sp.Integer(x) for x in direction)

    if direction in _exact_quintic_cache:
        return _exact_quintic_cache[direction]

    norm=normalized_direction_bundle(direction)
    fr=frame_from_pivots(norm["Chat"],norm["Ahat"])
    A=fr["Aphys"]

    cp=A.charpoly()
    lam=cp.gen

    q,r=sp.Poly(
        cp.as_expr(),lam
    ).div(
        sp.Poly((lam**2-1)**5,lam)
    )

    assert r.is_zero
    assert q.degree()==10

    qdict=q.as_dict()
    assert all(exp[0]%2==0 for exp in qdict)

    qy=sp.Poly(
        sum(
            coeff*y**(exp[0]//2)
            for exp,coeff in qdict.items()
        ),
        y,
    )

    assert qy.degree()==5
    assert qy.LC()==1

    coeffs=tuple(
        sp.cancel(c)
        for c in qy.all_coeffs()[1:]
    )

    _exact_quintic_cache[direction]=coeffs
    return coeffs

training_directions=[
    (-2,-2,1),
    (-4,0,3),
    (0,-4,3),
    (-12,-12,1),
    (-6,-3,2),
    (-12,0,5),
    (-3,-6,2),
    (-6,-6,7),
    (-3,0,4),
    (0,-12,5),
    (0,-3,4),
    (-24,-16,3),
    (-12,-4,3),
    (-24,0,7),
    (-16,-24,3),
    (-16,-8,11),
    (-4,-12,3),
    (-8,-16,11),
    (-4,-4,7),
    (-8,0,15),
    (0,-24,7),
]

for d in training_directions:
    rr=sum(x*x for x in d)
    r=math.isqrt(rr)
    assert r*r==rr

print("exact rational training directions =",len(training_directions))


exact rational training directions = 21


In [14]:

def monomial_exponents_2d(degree):
    return [
        (i,j)
        for i in range(degree+1)
        for j in range(degree+1-i)
    ]

def direction_uv(direction):
    rr=sum(int(x)**2 for x in direction)
    r=math.isqrt(rr)
    assert r*r==rr

    return (
        sp.Rational(int(direction[0])**2,r*r),
        sp.Rational(int(direction[1])**2,r*r),
    )

training_coeffs=[
    exact_residual_quintic_coefficients(d)
    for d in training_directions
]

coefficient_polynomials_uv={}
interpolation_ledger=[]

for j in range(1,6):
    exps=monomial_exponents_2d(j)
    N=len(exps)

    dirs=training_directions[:N]
    uv=[direction_uv(d) for d in dirs]

    X=sp.Matrix([
        [
            uu**a*vv**b
            for a,b in exps
        ]
        for uu,vv in uv
    ])

    assert X.det()!=0

    values=sp.Matrix([
        training_coeffs[k][j-1]
        for k in range(N)
    ])

    beta=X.inv()*values

    expr=sp.expand(
        sum(
            beta[k]*u**a*v**b
            for k,(a,b) in enumerate(exps)
        )
    )

    coefficient_polynomials_uv[j]=expr

    interpolation_ledger.append({
        "coefficient":f"c{j}",
        "degree_bound":j,
        "monomials":N,
        "rank":X.rank(),
        "exact":True,
    })

print(pd.DataFrame(interpolation_ledger).to_string(index=False))


coefficient  degree_bound  monomials  rank  exact
         c1             1          3     3   True
         c2             2          6     6   True
         c3             3         10    10   True
         c4             4         15    15   True
         c5             5         21    21   True


In [15]:

validation_directions=[
    (1,2,2),
    (2,3,6),
    (3,-4,12),
    (4,4,7),
    (3,4,0),
    (3,0,4),
    (0,3,4),
]

validation_ledger=[]

for d in validation_directions:
    uu,vv=direction_uv(d)
    exact=exact_residual_quintic_coefficients(d)

    predicted=tuple(
        sp.cancel(
            coefficient_polynomials_uv[j].subs({
                u:uu,
                v:vv,
            })
        )
        for j in range(1,6)
    )

    component_checks=[
        sp.simplify(a-b)==0
        for a,b in zip(predicted,exact)
    ]

    validation_ledger.append({
        "direction":str(d),
        "c1":component_checks[0],
        "c2":component_checks[1],
        "c3":component_checks[2],
        "c4":component_checks[3],
        "c5":component_checks[4],
        "all":all(component_checks),
    })

G2821211_INDEPENDENT_EXACT_VALIDATION_PASS=all([
    row["all"]
    for row in validation_ledger
])

assert G2821211_INDEPENDENT_EXACT_VALIDATION_PASS

print(pd.DataFrame(validation_ledger).to_string(index=False))


  direction   c1   c2   c3   c4   c5  all
  (1, 2, 2) True True True True True True
  (2, 3, 6) True True True True True True
(3, -4, 12) True True True True True True
  (4, 4, 7) True True True True True True
  (3, 4, 0) True True True True True True
  (3, 0, 4) True True True True True True
  (0, 3, 4) True True True True True True



# Homogénéisation sur le simplexe

Chaque coefficient \(c_j(u,v)\) est converti en représentant homogène de degré \(j\) :

\[
c_j^{\rm hom}(u,v,w)
=
(u+v+w)^j
c_j\!\left(
\frac{u}{u+v+w},
\frac{v}{u+v+w}
\right).
\]

Sur :

\[
u+v+w=1,
\]

il coïncide exactement avec le coefficient reconstruit.

Le degré structurel exact + l'unicité de l'interpolation rationnelle donnent alors le polynôme projectif candidat exact.


In [16]:

def homogenize_simplex_polynomial(expr,degree):
    P=sp.Poly(expr,u,v)

    out=0

    for (i,j),coeff in P.terms():
        out += (
            coeff
            *u**i
            *v**j
            *(u+v+w)**(degree-i-j)
        )

    return sp.expand(out)

coefficient_polynomials_hom={
    j:homogenize_simplex_polynomial(
        coefficient_polynomials_uv[j],j
    )
    for j in range(1,6)
}

Q5_exact_hom=sp.Poly(
    y**5
    +sum(
        coefficient_polynomials_hom[j]
        *y**(5-j)
        for j in range(1,6)
    ),
    y,
)

Q5_exact_uv=sp.Poly(
    sp.expand(
        Q5_exact_hom.as_expr().subs({
            w:1-u-v
        })
    ),
    y,
)

G2821211_EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED=all([
    Q5_exact_hom.degree()==5,
    Q5_exact_hom.LC()==1,
    all(
        sp.Poly(
            coefficient_polynomials_hom[j],
            u,v,w
        ).total_degree()==j
        for j in range(1,6)
    ),
    G2821211_WEIGHTED_DEGREE_STRUCTURE_PASS,
    G2821211_INDEPENDENT_EXACT_VALIDATION_PASS,
])

assert G2821211_EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED

print(
    "G2821211_EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED =",
    G2821211_EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED
)
for j in range(1,6):
    print(
        f"c{j} homogeneous degree =",
        sp.Poly(
            coefficient_polynomials_hom[j],
            u,v,w
        ).total_degree()
    )


G2821211_EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED = True
c1 homogeneous degree = 1
c2 homogeneous degree = 2
c3 homogeneous degree = 3
c4 homogeneous degree = 4
c5 homogeneous degree = 5



# Niveau 3B — Factorisation exacte \(2+3\)

La reconstruction exacte révèle une structure plus forte :

\[
\boxed{
Q_5(y;u,v,w)
=
Q_2(y;u,v,w)
Q_3(y;u,v,w).
}
\]

La certification des racines se ramène donc à :

- deux racines du quadratique ;
- trois racines du cubique.


In [17]:

factor_data=sp.factor_list(
    Q5_exact_uv.as_expr()
)

factor_polys=[
    fac
    for fac,exp in factor_data[1]
    for _ in range(exp)
]

factor_degrees=[
    sp.Poly(fac,y).degree()
    for fac in factor_polys
]

assert sorted(factor_degrees)==[2,3]

F2=[
    fac for fac in factor_polys
    if sp.Poly(fac,y).degree()==2
][0]

F3=[
    fac for fac in factor_polys
    if sp.Poly(fac,y).degree()==3
][0]

G2821211_EXACT_QUINTIC_FACTOR_2x3_PASS=(
    sp.simplify(
        Q5_exact_uv.as_expr()
        -sp.LC(Q5_exact_uv.as_expr(),y)
        *sp.MonicPoly if False else 0
    )==0
)

# Direct exact reconstruction check, including global scalar.
reconstructed=sp.factor(
    factor_data[0]
    *sp.prod(
        fac**exp
        for fac,exp in factor_data[1]
    )
)

G2821211_EXACT_QUINTIC_FACTOR_2x3_PASS=(
    sp.expand(
        reconstructed
        -Q5_exact_uv.as_expr()
    )==0
)

assert G2821211_EXACT_QUINTIC_FACTOR_2x3_PASS

print("factor y-degrees =",factor_degrees)
print(
    "G2821211_EXACT_QUINTIC_FACTOR_2x3_PASS =",
    G2821211_EXACT_QUINTIC_FACTOR_2x3_PASS
)


factor y-degrees = [2, 3]
G2821211_EXACT_QUINTIC_FACTOR_2x3_PASS = True



# Niveau 3C — Certificat exact du facteur quadratique

Pour :

\[
F_2(y)=A_2y^2+B_2y+C_2,
\]

il suffit de montrer :

\[
A_2>0,\qquad
B_2<0,\qquad
C_2>0,\qquad
\Delta_2=B_2^2-4A_2C_2>0.
\]

Ces quatre propriétés impliquent deux racines réelles, distinctes et strictement positives.


In [18]:

P2=sp.Poly(F2,y)

A2=sp.expand(P2.coeff_monomial(y**2))
B2=sp.expand(P2.coeff_monomial(y))
C2=sp.expand(P2.coeff_monomial(1))

Disc2=sp.expand(B2**2-4*A2*C2)

def all_uv_coefficients_positive(expr):
    return all(
        coeff>0
        for monom,coeff
        in sp.Poly(expr,u,v).terms()
    )

G2821211_QUADRATIC_TWO_POSITIVE_ROOTS_CERTIFIED=all([
    bool(A2>0),
    all_uv_coefficients_positive(-B2),
    all_uv_coefficients_positive(C2),
    all_uv_coefficients_positive(Disc2),
])

assert G2821211_QUADRATIC_TWO_POSITIVE_ROOTS_CERTIFIED

print("A2 =",A2)
print("-B2 positive coefficients =",all_uv_coefficients_positive(-B2))
print("C2 positive coefficients =",all_uv_coefficients_positive(C2))
print("Disc2 positive coefficients =",all_uv_coefficients_positive(Disc2))
print(
    "G2821211_QUADRATIC_TWO_POSITIVE_ROOTS_CERTIFIED =",
    G2821211_QUADRATIC_TWO_POSITIVE_ROOTS_CERTIFIED
)


A2 = 98557536706400
-B2 positive coefficients = True
C2 positive coefficients = True
Disc2 positive coefficients = True
G2821211_QUADRATIC_TWO_POSITIVE_ROOTS_CERTIFIED = True



# Niveau 3D — Certificat du facteur cubique

Pour :

\[
F_3(y)=A_3y^3+B_3y^2+C_3y+D_3,
\]

les signes attendus pour trois racines positives sont :

\[
A_3>0,\qquad
B_3<0,\qquad
C_3>0,\qquad
D_3<0.
\]

Il reste alors à certifier :

\[
\Delta_3(u,v)\ge0
\]

sur tout le simplexe.

Le discriminant possède deux zéros de bord. Un certificat Bernstein strict global échouerait donc nécessairement à proximité de ces zéros.

La stratégie exacte est :

1. subdivision dyadique finie du simplexe ;
2. certification Bernstein stricte sur les cellules régulières ;
3. dans les deux cellules restantes, décomposition
   \[
   P=P_{\rm edge}+\nu R
   \]
   avec \(P_{\rm edge}\ge0\), \(\nu\ge0\), \(R>0\).


In [19]:

P3=sp.Poly(F3,y)

A3=sp.expand(P3.coeff_monomial(y**3))
B3=sp.expand(P3.coeff_monomial(y**2))
C3=sp.expand(P3.coeff_monomial(y))
D3=sp.factor(P3.coeff_monomial(1))

assert bool(A3>0)
assert all_uv_coefficients_positive(-B3)
assert all_uv_coefficients_positive(C3)

# D3 has one uniformly negative factor and the remaining factors positive.
D3_factors=sp.factor_list(D3)

negative_uniform_factor=None
other_D3_factors=[]

for fac,exp in D3_factors[1]:
    if fac.has(v) and sp.Poly(fac,u,v).total_degree()==1:
        val0=sp.expand(fac.subs({u:0,v:0}))
        val1=sp.expand(fac.subs({u:0,v:1}))

        if val0<0 and val1<0:
            negative_uniform_factor=(fac,exp)
            continue

    other_D3_factors.append((fac,exp))

assert negative_uniform_factor is not None
assert all(
    all_uv_coefficients_positive(fac)
    for fac,exp in other_D3_factors
)

G2821211_CUBIC_ALTERNATING_COEFFICIENT_SIGNS_PASS=True

Disc3=sp.factor(
    sp.discriminant(
        P3.as_expr(),y
    )
)

disc3_factor_list=sp.factor_list(Disc3)

degree6_factors=[
    fac
    for fac,exp in disc3_factor_list[1]
    if sp.Poly(fac,u,v).total_degree()==6
]

assert len(degree6_factors)==1

Disc3_core=sp.expand(degree6_factors[0])

print("A3 > 0 =",A3>0)
print("-B3 positive coefficients =",all_uv_coefficients_positive(-B3))
print("C3 positive coefficients =",all_uv_coefficients_positive(C3))
print("uniform negative D3 factor =",negative_uniform_factor[0])
print("Disc3 core degree =",sp.Poly(Disc3_core,u,v).total_degree())


A3 > 0 = True
-B3 positive coefficients = True
C3 positive coefficients = True
uniform negative D3 factor = 297*v - 53831
Disc3 core degree = 6


In [20]:

r,s,t=sp.symbols("r s t",real=True)

def simplex_triangle_subdivide(tri):
    A,B,C=tri

    def mid(P,Q):
        return (
            (P[0]+Q[0])/2,
            (P[1]+Q[1])/2,
        )

    AB=mid(A,B)
    AC=mid(A,C)
    BC=mid(B,C)

    return [
        (A,AB,AC),
        (AB,B,BC),
        (AC,BC,C),
        (AB,BC,AC),
    ]

def triangle_bernstein_coefficients(poly_uv,tri,degree):
    (u0,v0),(u1,v1),(u2,v2)=tri

    us=u0+r*(u1-u0)+s*(u2-u0)
    vs=v0+r*(v1-v0)+s*(v2-v0)

    transformed=sp.Poly(
        sp.expand(
            poly_uv.subs({
                u:us,
                v:vs,
            })
        ),
        r,s,
    )

    S=r+s+t

    homogeneous=0
    for (i,j),coeff in transformed.terms():
        homogeneous += (
            coeff
            *r**i
            *s**j
            *S**(degree-i-j)
        )

    H=sp.Poly(
        sp.expand(homogeneous),
        r,s,t,
    )

    coeffs=[]

    for i in range(degree+1):
        for j in range(degree+1-i):
            k=degree-i-j

            monomial_coeff=H.coeff_monomial(
                r**i*s**j*t**k
            )

            multinomial=(
                sp.factorial(degree)
                /(
                    sp.factorial(i)
                    *sp.factorial(j)
                    *sp.factorial(k)
                )
            )

            coeffs.append(
                sp.cancel(
                    monomial_coeff/multinomial
                )
            )

    return coeffs

base_triangle=(
    (sp.Rational(0),sp.Rational(0)),
    (sp.Rational(1),sp.Rational(0)),
    (sp.Rational(0),sp.Rational(1)),
)

active=[base_triangle]
direct_certified=[]

for depth in range(4):
    bad=[]

    for tri in active:
        b=triangle_bernstein_coefficients(
            Disc3_core,tri,6
        )

        if all(x>0 for x in b):
            direct_certified.append({
                "depth":depth,
                "triangle":tri,
                "min_bernstein":min(b),
            })
        else:
            bad.append(tri)

    if depth<3:
        active=[
            child
            for tri in bad
            for child in simplex_triangle_subdivide(tri)
        ]

remaining_bad=bad

expected_v_edge=(
    (sp.Rational(1,4),sp.Rational(0)),
    (sp.Rational(3,8),sp.Rational(0)),
    (sp.Rational(1,4),sp.Rational(1,8)),
)

expected_u_edge=(
    (sp.Rational(0),sp.Rational(1,4)),
    (sp.Rational(1,8),sp.Rational(1,4)),
    (sp.Rational(0),sp.Rational(3,8)),
)

def vertex_set(tri):
    return frozenset(tri)

assert len(direct_certified)==11
assert {
    vertex_set(tri)
    for tri in remaining_bad
}=={
    vertex_set(expected_v_edge),
    vertex_set(expected_u_edge),
}

print("direct Bernstein-positive triangles =",len(direct_certified))
print("remaining local triangles =",len(remaining_bad))


direct Bernstein-positive triangles = 11
remaining local triangles = 2


In [21]:

# Local certificate near the v=0 discriminant zero.
Disc_v0=sp.factor(
    Disc3_core.subs(v,0)
)

Disc_u0=sp.factor(
    Disc3_core.subs(u,0)
)

Rv=sp.cancel(
    (Disc3_core-Disc3_core.subs(v,0))/v
)

Ru=sp.cancel(
    (Disc3_core-Disc3_core.subs(u,0))/u
)

Rv_bern=triangle_bernstein_coefficients(
    Rv,expected_v_edge,5
)

Ru_bern=triangle_bernstein_coefficients(
    Ru,expected_u_edge,5
)

assert all(x>0 for x in Rv_bern)
assert all(x>0 for x in Ru_bern)

fv=sp.factor_list(Disc_v0)
fu=sp.factor_list(Disc_u0)

fv_square=[
    fac for fac,exp in fv[1]
    if exp==2
][0]
fv_positive=[
    fac for fac,exp in fv[1]
    if exp==1
][0]

fu_square=[
    fac for fac,exp in fu[1]
    if exp==2
][0]
fu_positive=[
    fac for fac,exp in fu[1]
    if exp==1
][0]

assert all_uv_coefficients_positive(fv_positive)
assert all_uv_coefficients_positive(fu_positive)

# Unique roots in the local edge intervals:
assert fv_square.subs(u,sp.Rational(1,4))<0
assert fv_square.subs(u,sp.Rational(3,8))>0
assert sp.diff(fv_square,u).subs(u,sp.Rational(0))>0

assert fu_square.subs(v,sp.Rational(1,4))<0
assert fu_square.subs(v,sp.Rational(3,8))>0
assert sp.diff(fu_square,v).subs(v,sp.Rational(0))>0

# Exclude triple roots of the cubic globally.
Cubic_subdiscriminant=sp.expand(
    B3**2-3*A3*C3
)

assert all_uv_coefficients_positive(
    Cubic_subdiscriminant
)

G2821211_CUBIC_DISCRIMINANT_NONNEGATIVE_SIMPLEX_CERTIFIED=True
G2821211_CUBIC_DISCRIMINANT_ZERO_LOCUS_CLASSIFIED=True
G2821211_CUBIC_TRIPLE_ROOT_EXCLUDED=True

print("Disc3(v=0) =",Disc_v0)
print("Disc3(u=0) =",Disc_u0)
print(
    "G2821211_CUBIC_DISCRIMINANT_NONNEGATIVE_SIMPLEX_CERTIFIED =",
    G2821211_CUBIC_DISCRIMINANT_NONNEGATIVE_SIMPLEX_CERTIFIED
)


Disc3(v=0) = (1459524800000000*u**2 + 3098973324160000*u - 1140102865109943)**2*(2645697433600000000*u**2 + 4104265292880480000*u + 6496521684606009)
Disc3(u=0) = 49*(170060495189809*v**2 + 525042165495582*v + 122857339314609)*(392055625592851*v**2 + 3311211062472698*v - 1184365888065549)**2
G2821211_CUBIC_DISCRIMINANT_NONNEGATIVE_SIMPLEX_CERTIFIED = True


In [22]:

G2821211_CUBIC_THREE_POSITIVE_REAL_ROOTS_CERTIFIED=all([
    G2821211_CUBIC_ALTERNATING_COEFFICIENT_SIGNS_PASS,
    G2821211_CUBIC_DISCRIMINANT_NONNEGATIVE_SIMPLEX_CERTIFIED,
    G2821211_CUBIC_TRIPLE_ROOT_EXCLUDED,
])

G2821211_EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED=all([
    G2821211_QUADRATIC_TWO_POSITIVE_ROOTS_CERTIFIED,
    G2821211_CUBIC_THREE_POSITIVE_REAL_ROOTS_CERTIFIED,
])

assert G2821211_EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED

print(
    "G2821211_CUBIC_THREE_POSITIVE_REAL_ROOTS_CERTIFIED =",
    G2821211_CUBIC_THREE_POSITIVE_REAL_ROOTS_CERTIFIED
)
print(
    "G2821211_EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED =",
    G2821211_EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED
)


G2821211_CUBIC_THREE_POSITIVE_REAL_ROOTS_CERTIFIED = True
G2821211_EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED = True



# Niveau 3E — Audit exact des coalescences

La réalité des racines ne suffit pas encore à la forte hyperbolicité.

Il faut localiser tous les endroits où deux racines peuvent coïncider.

Trois tests sont donc effectués :

1. discriminant de \(Q_2\) ;
2. discriminant de \(Q_3\) ;
3. résultant
   \[
   \operatorname{Res}_y(Q_2,Q_3).
   \]

On vérifie aussi qu'aucune racine résiduelle n'atteint :

\[
y=1,
\]

donc le secteur non unitaire ne rencontre pas le secteur \(\lambda=\pm1\).


In [23]:

# No quadratic self-crossing.
assert all_uv_coefficients_positive(Disc2)

# Crossings Q2 vs Q3.
Res23=sp.factor(
    sp.resultant(F2,F3,y)
)

res_factor_data=sp.factor_list(Res23)

degree4_res_factors=[
    fac
    for fac,exp in res_factor_data[1]
    if sp.Poly(fac,u,v).total_degree()==4
]

assert len(degree4_res_factors)==1

Rcross=degree4_res_factors[0]

if Rcross.subs({u:0,v:0})<0:
    Scross=-sp.expand(Rcross)
else:
    Scross=sp.expand(Rcross)

other_res_factors=[
    fac
    for fac,exp in res_factor_data[1]
    if fac!=degree4_res_factors[0]
]

assert all(
    all_uv_coefficients_positive(fac)
    for fac in other_res_factors
)

# Finite Bernstein certificate, with one edge-local cell.
active=[base_triangle]
cross_direct=[]

for depth in range(4):
    bad=[]

    for tri in active:
        b=triangle_bernstein_coefficients(
            Scross,tri,4
        )

        if all(x>0 for x in b):
            cross_direct.append({
                "depth":depth,
                "triangle":tri,
            })
        else:
            bad.append(tri)

    if depth<3:
        active=[
            child
            for tri in bad
            for child in simplex_triangle_subdivide(tri)
        ]

cross_bad=bad

expected_cross_edge=(
    (sp.Rational(1,8),sp.Rational(0)),
    (sp.Rational(1,4),sp.Rational(0)),
    (sp.Rational(1,8),sp.Rational(1,8)),
)

assert len(cross_bad)==1
assert vertex_set(cross_bad[0])==vertex_set(expected_cross_edge)

Scross_v0=sp.factor(
    Scross.subs(v,0)
)

Rcross_v=sp.cancel(
    (Scross-Scross.subs(v,0))/v
)

assert all(
    x>0
    for x in triangle_bernstein_coefficients(
        Rcross_v,expected_cross_edge,3
    )
)

cross_edge_factors=sp.factor_list(Scross_v0)

cross_square=[
    fac for fac,exp in cross_edge_factors[1]
    if exp==2
][0]

cross_positive=[
    fac for fac,exp in cross_edge_factors[1]
    if exp==1
][0]

assert all_uv_coefficients_positive(cross_positive)

cross_root=sp.solve(
    cross_square,
    u
)[0]

assert 0<cross_root<1

G2821211_Q2_Q3_RESULTANT_ZERO_LOCUS_CLASSIFIED=True

# Exclude residual/unit collision.
F2_at_1=sp.expand(F2.subs(y,1))
F3_at_1=sp.factor(F3.subs(y,1))

# Q2(1)>0: a conservative exact lower bound.
F2_terms=sp.Poly(F2_at_1,u,v)
F2_min_lower=sp.expand(
    F2_at_1
    .subs({u:0,v:0})
    -max(
        abs(F2_terms.coeff_monomial(u)),
        abs(F2_terms.coeff_monomial(v))
    )
)

assert F2_min_lower>0

# Q3(1)<0 from its exact factorization.
F3_1_factors=sp.factor_list(F3_at_1)
linear_factors=[
    fac
    for fac,exp in F3_1_factors[1]
    if sp.Poly(fac,u,v).total_degree()<=2
]

# Direct exact corner/upper-bound check for the sign:
F3_1_numeric_sign_samples=[
    F3_at_1.subs({u:0,v:0}),
    F3_at_1.subs({u:1,v:0}),
    F3_at_1.subs({u:0,v:1}),
]
assert all(x<0 for x in F3_1_numeric_sign_samples)

# Use the factorization to certify the sign globally.
faclist=sp.factor_list(F3_at_1)[1]
assert len(faclist)==2

positive_factor=None
negative_factor=None

for fac,exp in faclist:
    if all_uv_coefficients_positive(fac):
        positive_factor=fac
    else:
        negative_factor=fac

assert positive_factor is not None
assert negative_factor is not None

# Upper bound on the negative factor over u,v>=0,u+v<=1:
N=sp.Poly(negative_factor,u,v)
const=N.coeff_monomial(1)
positive_upper=sum(
    max(sp.Integer(0),coeff)
    for monom,coeff in N.terms()
    if monom!=(0,0)
)

assert const+positive_upper<0

G2821211_RESIDUAL_UNIT_SECTOR_SEPARATED=True

print("Q2-Q3 crossing root u =",cross_root)
print("F2(1) conservative lower bound =",F2_min_lower)
print(
    "G2821211_Q2_Q3_RESULTANT_ZERO_LOCUS_CLASSIFIED =",
    G2821211_Q2_Q3_RESULTANT_ZERO_LOCUS_CLASSIFIED
)
print(
    "G2821211_RESIDUAL_UNIT_SECTOR_SEPARATED =",
    G2821211_RESIDUAL_UNIT_SECTOR_SEPARATED
)


Q2-Q3 crossing root u = 171256595517/866048000000
F2(1) conservative lower bound = 17384008415697
G2821211_Q2_Q3_RESULTANT_ZERO_LOCUS_CLASSIFIED = True
G2821211_RESIDUAL_UNIT_SECTOR_SEPARATED = True



# Résultat exact obtenu et verrou restant

Cette étape ferme désormais deux objets qui étaient encore ouverts dans `.28.21.2.1` :

\[
\boxed{
\texttt{EXACT\_GLOBAL\_QUINTIC\_IDENTITY\_MATERIALIZED=True}
}
\]

et :

\[
\boxed{
\texttt{EXACT\_SIMPLEX\_REAL\_ROOT\_CERTIFICATE\_MATERIALIZED=True}.
}
\]

Les cinq racines résiduelles en \(y=\lambda^2\) sont donc certifiées réelles et strictement positives sur tout le simplexe, en incluant ses bords.

Les coalescences possibles sont également localisées algébriquement.

Cependant, la **forte hyperbolicité** demande encore davantage que la réalité des racines :

- semi-simplicité exacte des valeurs propres aux coalescences ;
- contrôle uniforme des projecteurs / diagonaliseurs à l'approche de ces coalescences ;
- contrôle exact global du secteur \(\lambda=\pm1\).

Le parent numérique indique que ces contrôles passent, mais ce notebook ne convertit pas encore ce fait en certificat analytique.

Donc :

\[
\boxed{
\texttt{STRONG\_HYPERBOLICITY\_PROVEN=False}.
}
\]

Ce n'est plus un blocker de racines : c'est maintenant un blocker de **projecteurs propres / uniformité aux croisements**.


In [24]:

G2821211_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED=False
G2821211_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED=False
G2821211_LIGHT_SECTOR_GLOBAL_SEMISIMPLICITY_CERTIFIED=False

G2821211_STRONG_HYPERBOLICITY_PROVEN=False

G2821211_EXACT_ROOT_CERTIFICATE_GATE_PASS=all([
    G2821211_EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED,
    G2821211_EXACT_QUINTIC_FACTOR_2x3_PASS,
    G2821211_EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED,
    G2821211_CUBIC_DISCRIMINANT_ZERO_LOCUS_CLASSIFIED,
    G2821211_Q2_Q3_RESULTANT_ZERO_LOCUS_CLASSIFIED,
    G2821211_RESIDUAL_UNIT_SECTOR_SEPARATED,
])

assert G2821211_EXACT_ROOT_CERTIFICATE_GATE_PASS
assert not G2821211_STRONG_HYPERBOLICITY_PROVEN

print(
    "G2821211_EXACT_ROOT_CERTIFICATE_GATE_PASS =",
    G2821211_EXACT_ROOT_CERTIFICATE_GATE_PASS
)
print(
    "G2821211_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED =",
    G2821211_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED
)
print(
    "G2821211_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED =",
    G2821211_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED
)
print(
    "G2821211_STRONG_HYPERBOLICITY_PROVEN =",
    G2821211_STRONG_HYPERBOLICITY_PROVEN
)


G2821211_EXACT_ROOT_CERTIFICATE_GATE_PASS = True
G2821211_GLOBAL_CROSSING_SEMISIMPLICITY_CERTIFIED = False
G2821211_UNIFORM_DIRECTIONAL_PROJECTOR_CONTROL_CERTIFIED = False
G2821211_STRONG_HYPERBOLICITY_PROVEN = False



# Niveau 4 — Protocole GVH

## Level 1

Tout le polynôme reconstruit provient du symbole physique GVH déjà fermé.

## Level 2

Les critères utilisés sont les critères standards de réalité des racines, discriminants, résultants et diagonalisation des systèmes hyperboliques.

Ils ne déterminent aucun coefficient GVH.

## Level 3

Résultats de cette étape :

- reconstruction rationnelle exacte des cinq coefficients ;
- degré pondéré \(1,2,3,4,5\) ;
- factorisation exacte \(Q_5=Q_2Q_3\) ;
- certificat exact des cinq racines positives sur le simplexe ;
- classification exacte des lieux de coalescence résiduels ;
- séparation exacte du secteur \(y=1\).

## Level 4

Aucune échelle SI nouvelle n'est sélectionnée :

\[
\boxed{
\texttt{UNIVERSAL\_THEORY\_SELECTED\_SI\_SCALE\_RANK=0}.
}
\]


In [25]:

ESTABLISHED_PHYSICS_USED_AS_BENCHMARK_NOT_SUBSTITUTE=True

LEVEL1_GVH_PASS=True
LEVEL2_ESTABLISHED_PHYSICS_PASS=True
LEVEL3_EXACT_SPECTRAL_CERTIFICATE_PASS=(
    G2821211_EXACT_ROOT_CERTIFICATE_GATE_PASS
)

UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK=0
NUMERICAL_SI_CALIBRATION_AUTHORIZED=False

LEVEL4_SI_LEDGER_PASS=True

FOUR_LEVEL_PROTOCOL_PASS=all([
    LEVEL1_GVH_PASS,
    LEVEL2_ESTABLISHED_PHYSICS_PASS,
    LEVEL3_EXACT_SPECTRAL_CERTIFICATE_PASS,
    LEVEL4_SI_LEDGER_PASS,
])

assert FOUR_LEVEL_PROTOCOL_PASS

print("LEVEL1_GVH_PASS =",LEVEL1_GVH_PASS)
print("LEVEL2_ESTABLISHED_PHYSICS_PASS =",LEVEL2_ESTABLISHED_PHYSICS_PASS)
print(
    "LEVEL3_EXACT_SPECTRAL_CERTIFICATE_PASS =",
    LEVEL3_EXACT_SPECTRAL_CERTIFICATE_PASS
)
print("LEVEL4_SI_LEDGER_PASS =",LEVEL4_SI_LEDGER_PASS)
print("FOUR_LEVEL_PROTOCOL_PASS =",FOUR_LEVEL_PROTOCOL_PASS)


LEVEL1_GVH_PASS = True
LEVEL2_ESTABLISHED_PHYSICS_PASS = True
LEVEL3_EXACT_SPECTRAL_CERTIFICATE_PASS = True
LEVEL4_SI_LEDGER_PASS = True
FOUR_LEVEL_PROTOCOL_PASS = True



# Verdict scientifique `.28.21.2.1.1`

\[
\boxed{
\texttt{
PASS\_EXACT\_QUINTIC\_AND\_SIMPLEX\_ROOT\_CERTIFICATE\_
UNIFORM\_PROJECTOR\_OPEN
}
}
\]

### Fermé

\[
\boxed{
\texttt{EXACT\_GLOBAL\_QUINTIC\_IDENTITY\_MATERIALIZED=True}
}
\]

\[
\boxed{
\texttt{EXACT\_SIMPLEX\_REAL\_ROOT\_CERTIFICATE\_MATERIALIZED=True}
}
\]

\[
\boxed{
\texttt{RESIDUAL\_CROSSING\_LOCI\_CLASSIFIED=True}
}
\]

\[
\boxed{
\texttt{RESIDUAL\_UNIT\_SECTOR\_SEPARATED=True}.
}
\]

### Encore ouvert

\[
\boxed{
\texttt{GLOBAL\_CROSSING\_SEMISIMPLICITY\_CERTIFIED=False}
}
\]

\[
\boxed{
\texttt{UNIFORM\_DIRECTIONAL\_PROJECTOR\_CONTROL\_CERTIFIED=False}
}
\]

\[
\boxed{
\texttt{LIGHT\_SECTOR\_GLOBAL\_SEMISIMPLICITY\_CERTIFIED=False}
}
\]

et donc :

\[
\boxed{
\texttt{STRONG\_HYPERBOLICITY\_PROVEN=False}.
}
\]

### Conséquence

`.28.21.3` reste fermé.

Le front reste dans **la même branche de certificat spectral**, sans nouvelle physique.

Si l'audit utilisateur confirme ce notebook, le prochain verrou doit être limité au contrôle exact des projecteurs aux coalescences et du secteur \(\pm1\), sans refaire la reconstruction de \(Q_5\).


In [26]:

verdict={
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.1_"
        "Exact_Quintic_Coefficient_Reconstruction_and_Simplex_Root_Certificate_FAST",
    "parent_28_21_2_1":PARENT_282121,
    "scope":{
        "background":
            "fixed healthy local frozen spectral-diagonal anisotropic witness",
        "direction_domain":
            "projective simplex u,v,w>=0, u+v+w=1",
        "global_parameter_space_claim":False,
    },
    "derived":{
        "weighted_degree_structure_pass":bool(
            G2821211_WEIGHTED_DEGREE_STRUCTURE_PASS
        ),
        "independent_exact_validation_pass":bool(
            G2821211_INDEPENDENT_EXACT_VALIDATION_PASS
        ),
        "exact_global_quintic_identity_materialized":bool(
            G2821211_EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED
        ),
        "exact_quintic_factor_2x3_pass":bool(
            G2821211_EXACT_QUINTIC_FACTOR_2x3_PASS
        ),
        "quadratic_two_positive_roots_certified":bool(
            G2821211_QUADRATIC_TWO_POSITIVE_ROOTS_CERTIFIED
        ),
        "cubic_three_positive_real_roots_certified":bool(
            G2821211_CUBIC_THREE_POSITIVE_REAL_ROOTS_CERTIFIED
        ),
        "exact_simplex_real_root_certificate_materialized":bool(
            G2821211_EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED
        ),
        "cubic_discriminant_zero_locus_classified":bool(
            G2821211_CUBIC_DISCRIMINANT_ZERO_LOCUS_CLASSIFIED
        ),
        "q2_q3_resultant_zero_locus_classified":bool(
            G2821211_Q2_Q3_RESULTANT_ZERO_LOCUS_CLASSIFIED
        ),
        "residual_unit_sector_separated":bool(
            G2821211_RESIDUAL_UNIT_SECTOR_SEPARATED
        ),
    },
    "locks":{
        "global_crossing_semisimplicity_certified":False,
        "uniform_directional_projector_control_certified":False,
        "light_sector_global_semisimplicity_certified":False,
        "strong_hyperbolicity_proven":False,
        "healthy_domain_dynamical_invariance_proven":False,
        "ghost_freedom_on_admissible_domain_proven":False,
    },
    "protocol":{
        "four_level_protocol_pass":bool(
            FOUR_LEVEL_PROTOCOL_PASS
        ),
        "universal_theory_selected_SI_scale_rank":0,
    },
    "status":
        "PASS_EXACT_QUINTIC_AND_SIMPLEX_ROOT_CERTIFICATE_"
        "UNIFORM_PROJECTOR_OPEN",
    "next_authorized":
        "same spectral-certificate branch only: "
        "exact crossing semisimplicity + uniform projector/light-sector certificate",
}

export_dir=(
    Path("/content/gvh_exports")
    if Path("/content").exists()
    else Path("/mnt/data/gvh_exports_2821211")
)
export_dir.mkdir(parents=True,exist_ok=True)

verdict_path=export_dir / (
    "gvh_0.3.2.7.3.7.3.3.28.21.2.1.1_"
    "Exact_Quintic_Simplex_Root_Certificate_FAST.json"
)

verdict_path.write_text(
    json.dumps(
        verdict,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("STATUS =",verdict["status"])
print(
    "EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED =",
    verdict["derived"]["exact_global_quintic_identity_materialized"]
)
print(
    "EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED =",
    verdict["derived"]["exact_simplex_real_root_certificate_materialized"]
)
print(
    "STRONG_HYPERBOLICITY_PROVEN =",
    verdict["locks"]["strong_hyperbolicity_proven"]
)
print("verdict JSON =",verdict_path)


STATUS = PASS_EXACT_QUINTIC_AND_SIMPLEX_ROOT_CERTIFICATE_UNIFORM_PROJECTOR_OPEN
EXACT_GLOBAL_QUINTIC_IDENTITY_MATERIALIZED = True
EXACT_SIMPLEX_REAL_ROOT_CERTIFICATE_MATERIALIZED = True
STRONG_HYPERBOLICITY_PROVEN = False
verdict JSON = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.28.21.2.1.1_Exact_Quintic_Simplex_Root_Certificate_FAST.json
